# Create Croucher Foundation Awards (Hong Kong)

Creates Croucher Foundation scholarships, fellowships and other awards from the
foundation's own directory of Croucher fellows and scholars on its main site,
https://croucher.org.hk/en/fellows-and-scholars (Next.js; the people and their award
lists are embedded as JSON in each listing page). **1,479 awards to 1,304 people,
1981-2026** (local run 2026-10-01, oxjob #1451): Scholarship 592, Fellowship 439,
Studentship 175, Senior Research Fellowship 116, Innovation Award 40, Cambridge
International Scholarship 27, Oxford Croucher Scholarship 25, Senior Medical Research
Fellowship 25, Butterfield Croucher Studentship 13, MBBS/PhD Scholarship 10, and 8 rarer
types. One row per (person x award); a scholar who later won a fellowship has two rows.
Two listed people with no award entry are skipped.

**robots:** croucher.org.hk allows all agents. The separate directory host
scholars.croucher.org.hk (the tracker's original source URL) disallows all non-Google
agents and was not used.

**Scope:** all award types kept (talent/training pathways are in scope). `funding_type`
= `fellowship` for scholarships, studentships, fellowships, professorships and the
visitorship; `research` for the Croucher Innovation Award (a research grant to early-career
HK academics).

**Fields:** the directory gives name (+ Chinese name), award year, award type and the
host institution where the award is held. No amounts, dates or grant numbers are
published per award: `amount`/`currency` NULL (§6.7 waived), `start_year` = award year,
`start_date`/`end_date` NULL. `display_name` = "{award type} {year}: {name}".
`lead_investigator` = the scholar/fellow, affiliation = the host institution (the
institution printed in the award). Names are printed given-first and split with the
canonical §2.4.1 `split_name` (last token = family name; "Ho Wan Cheng" -> Cheng).

**Prerequisites:** run `scripts/local/croucher_foundation_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/croucher_foundation/croucher_foundation_projects.parquet`
(§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** Croucher publishes no award numbers. Citing works carry
mostly host-university project codes; one Croucher-internal number appears
(`CIA14SC04`, an Innovation Award) but the directory does not expose them. So this ships
a stable synthetic key `CROUCHER-{year}-{award code}-{person slug}`
(e.g. `CROUCHER-2026-fll-leo-wang`); 0 existing stubs collapse.

**Twin:** `CAS-Croucher Funding Scheme for Joint Laboratories` (F4320326784, 91 stub rows)
is a separate Croucher/CAS scheme not covered by this directory; nothing here is assigned
to it.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320320904`
- display_name: Croucher Foundation
- ror_id / doi: from `openalex.funders.funders` (ror 04741mc34, doi 10.13039/501100001692)

**Priority:** `542` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.croucher_foundation_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/croucher_foundation/croucher_foundation_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.croucher_foundation_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.croucher_foundation_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.croucher_foundation_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320320904 is a Crossref-registered F4320* funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320320904;

## Step 2: Create Croucher Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.croucher_foundation_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320320904  -- Croucher Foundation
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    CONCAT(g.funder_scheme, ' ', g.award_year, ': ', TRIM(g.name)) as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,      -- not published per award
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type as funding_type,
    g.funder_scheme as funder_scheme,
    'croucher_fellows_and_scholars' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.award_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN NULLIF(TRIM(g.family_name), '') IS NOT NULL THEN
        named_struct(
            'given_name', NULLIF(TRIM(g.given_name), ''),
            'family_name', NULLIF(TRIM(g.family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        )
    END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN NULLIF(TRIM(g.family_name), '') IS NOT NULL THEN array(
        named_struct(
            'given_name', NULLIF(TRIM(g.given_name), ''),
            'family_name', NULLIF(TRIM(g.family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        )
    ) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.croucher_foundation_raw g
CROSS JOIN src_funder f
WHERE NULLIF(TRIM(g.funder_award_id), '') IS NOT NULL;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'croucher_fellows_and_scholars' AND priority = 542;

-- Insert into openalex_awards_raw with priority.
-- Priority 542: direct-from-funder ingest (higher wins under the 2026-06-20 DESC
-- dedup, oxjob #500), so it outranks the priority-0 acknowledgement shells that
-- share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    542 as priority
FROM openalex.awards.croucher_foundation_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids,
       COUNT(DISTINCT LOWER(funder_award_id)) as distinct_award_ids
FROM openalex.awards.croucher_foundation_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, lead_investigator.affiliation.country
FROM openalex.awards.croucher_foundation_awards LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage
SELECT
    COUNT(*) as total,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    collect_set(currency) as currencies,
    MIN(amount) as min_amt, MAX(amount) as max_amt, ROUND(AVG(amount), 0) as avg_amt,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_start_date,
    ROUND(COUNT(start_year) * 100.0 / COUNT(*), 1) as pct_start_year,
    ROUND(COUNT(lead_investigator.family_name) * 100.0 / COUNT(*), 1) as pct_pi,
    ROUND(COUNT(lead_investigator.affiliation.name) * 100.0 / COUNT(*), 1) as pct_institution
FROM openalex.awards.croucher_foundation_awards;

In [ ]:
%sql
SELECT funder_scheme, currency, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_amount
FROM openalex.awards.croucher_foundation_awards
GROUP BY funder_scheme, currency ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.croucher_foundation_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.croucher_foundation_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.croucher_foundation_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.5 funder consistency
SELECT funder.display_name, funder_id, COUNT(*)
FROM openalex.awards.croucher_foundation_awards
GROUP BY funder.display_name, funder_id ORDER BY 3 DESC;

In [ ]:
%sql
-- Award-id shape (2.1.1): every row must be the documented synthetic key.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^CROUCHER-[0-9]{4}-[a-z]+-[a-z0-9-]+$' THEN 1 ELSE 0 END) AS synthetic_key,
    SUM(CASE WHEN NOT funder_award_id RLIKE '^CROUCHER-[0-9]{4}-[a-z]+-[a-z0-9-]+$' THEN 1 ELSE 0 END) AS other_shape -- must be 0
FROM openalex.awards.croucher_foundation_awards;

In [ ]:
%sql
SELECT funding_type, COUNT(*) AS n FROM openalex.awards.croucher_foundation_awards GROUP BY 1;

In [ ]:
%sql
-- Section 6.8: rows reached the shared raw table at the assigned priority.
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'croucher_fellows_and_scholars'
GROUP BY provenance, priority;